# NASA Exoplanet Archive: Data Collection

This notebook queries the **NASA Exoplanet Archive TAP API** to download the complete **Planetary Systems Composite Parameters (`pscomppars`)** dataset and saves it locally as `exoplanet_data.csv`.

### Overview:
* **Endpoint**: `https://exoplanetarchive.ipac.caltech.edu/TAP/sync`
* **Table**: `pscomppars` (compiles one comprehensive record per confirmed exoplanet across all published literature)
* **Format**: CSV

In [1]:
import os
from io import StringIO
import requests
import pandas as pd

print("Libraries successfully imported!")

Libraries successfully imported!


## 1. Request Data from NASA TAP API
We send an ADQL query (`SELECT * FROM pscomppars`) to the synchronous TAP endpoint.

In [2]:
# NASA Exoplanet Archive TAP API endpoint
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

# ADQL query parameters
params = {
    "query": "SELECT * FROM pscomppars",
    "format": "csv"
}

print("Fetching exoplanet dataset from NASA TAP API...")
try:
    # Set timeout=120 to reliably handle full ~80MB payload
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    
    payload_size_mb = len(response.content) / (1024 * 1024)
    print(f"Data successfully downloaded!")
    print(f"HTTP Status Code: {response.status_code}")
    print(f"Payload Size: {payload_size_mb:.2f} MB")
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
    raise

Fetching exoplanet dataset from NASA TAP API...
Data successfully downloaded!
HTTP Status Code: 200
Payload Size: 79.01 MB


## 2. Parse and Save Dataset to CSV
Convert the raw CSV text to a pandas DataFrame and save it as `exoplanet_data.csv`.

In [3]:
# Convert response to DataFrame
df = pd.read_csv(StringIO(response.text), low_memory=False)

# Save dataset
output_file = "exoplanet_data.csv"
df.to_csv(output_file, index=False)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Dataset saved successfully to '{output_file}' ({file_size_mb:.2f} MB)")
print(f"Total Rows (Confirmed Planets): {len(df)}")
print(f"Total Columns: {len(df.columns)}")

Dataset saved successfully to 'exoplanet_data.csv' (70.53 MB)
Total Rows (Confirmed Planets): 6445
Total Columns: 703


## 3. Data Preview & Inspection
Inspect the first few rows and inspect key physical and orbital columns.

In [4]:
# Preview first 5 rows
df.head()

,systemid,sy_name,objectid,pl_name,pl_letter,hostid,hostname,hd_name,hip_name,tic_id,...,pl_tsm_solnid,pl_tsmstr,pl_tsmlim,pl_esm,pl_esmerr1,pl_esmerr2,pl_esm_reflink,pl_esm_solnid,pl_esmstr,pl_esmlim
0,1.539830,HD 2039,3.11204,HD 2039 b,b,2.539830,HD 2039,HD 2039,HIP 1931,TIC 281461362,...,NaN,NaN,NaN,0.1,NaN,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,NaN,0.0,0.0
1,1.103446,HAT-P-8,3.11027,HAT-P-8 b,b,2.103446,HAT-P-8,NaN,NaN,TIC 188876052,...,NaN,109.0,0.0,120.0,NaN,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,NaN,120.0,0.0
2,1.643080,K2-43,3.88640,K2-43 b,b,2.643080,K2-43,NaN,NaN,TIC 443616612,...,NaN,60.0,0.0,14.5,NaN,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,NaN,15.0,0.0
3,1.567687,Kepler-1753,3.15620,Kepler-1753 b,b,2.567687,Kepler-1753,NaN,NaN,TIC 158664331,...,NaN,4.0,0.0,0.1,NaN,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,NaN,0.0,0.0
4,1.544946,Kepler-1176,3.24120,Kepler-1176 b,b,2.544946,Kepler-1176,NaN,NaN,TIC 273582656,...,NaN,3.0,0.0,0.1,NaN,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,NaN,0.0,0.0


In [5]:
# Inspect core physical, orbital, and stellar columns
key_columns = [
    "pl_name",          # Planet name
    "hostname",         # Host star name
    "discoverymethod",  # Detection method
    "disc_year",        # Discovery year
    "pl_orbper",        # Orbital period [days]
    "pl_orbsmax",       # Semi-major axis [AU]
    "pl_rade",          # Planet radius [Earth Radii]
    "pl_bmasse",        # Planet mass [Earth Masses]
    "pl_dens",          # Planet bulk density [g/cm^3]
    "pl_eqt",           # Planet equilibrium temperature [K]
    "st_teff",          # Stellar effective temperature [K]
    "st_rad",           # Stellar radius [Solar Radii]
    "st_mass",          # Stellar mass [Solar Masses]
    "st_met",           # Stellar metallicity [dex]
    "sy_dist"           # Distance to system [pc]
]

available_key_cols = [col for col in key_columns if col in df.columns]
print(f"Found {len(available_key_cols)} / {len(key_columns)} key columns in dataset.")
df[available_key_cols].info()

Found 15 / 15 key columns in dataset.
<class 'pandas.DataFrame'>
RangeIndex: 6445 entries, 0 to 6444
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pl_name          6445 non-null   str    
 1   hostname         6445 non-null   str    
 2   discoverymethod  6445 non-null   str    
 3   disc_year        6445 non-null   int64  
 4   pl_orbper        6092 non-null   float64
 5   pl_orbsmax       5953 non-null   float64
 6   pl_rade          6395 non-null   float64
 7   pl_bmasse        6414 non-null   float64
 8   pl_dens          6297 non-null   float64
 9   pl_eqt           5909 non-null   float64
 10  st_teff          6138 non-null   float64
 11  st_rad           6114 non-null   float64
 12  st_mass          6436 non-null   float64
 13  st_met           5747 non-null   float64
 14  sy_dist          6417 non-null   float64
dtypes: float64(11), int64(1), str(3)
memory usage: 942.5 KB


In [6]:
# Summary statistics for numerical key properties
df[available_key_cols].describe().round(3)

,disc_year,pl_orbper,pl_orbsmax,pl_rade,pl_bmasse,pl_dens,pl_eqt,st_teff,st_rad,st_mass,st_met,sy_dist
count,6445.000,6.092000e+03,5953.000,6395.000,6414.000,6297.000,5909.000,6138.000,6114.000,6436.000,5747.000,6417.000
mean,2017.395,7.027348e+04,15.422,5.846,401.339,4.814,836.633,5389.238,1.482,0.933,0.017,705.230
std,5.198,5.152872e+06,345.542,5.411,1133.324,33.776,496.280,1719.835,3.834,0.386,0.189,1230.682
min,1992.000,9.100000e-02,0.004,0.310,0.020,0.005,1.480,415.000,0.012,0.009,-1.000,1.301
25%,2014.000,4.285000e+00,0.052,1.850,4.320,1.270,486.000,4893.250,0.770,0.770,-0.080,99.556
50%,2016.000,1.064100e+01,0.103,2.860,9.320,2.510,758.000,5544.000,0.950,0.940,0.020,349.247
75%,2021.000,3.800000e+01,0.313,11.900,188.076,4.530,1095.000,5900.000,1.250,1.086,0.130,817.584
max,2026.000,4.020000e+08,19000.000,87.206,9534.852,2000.000,7777.780,57000.000,88.475,9.100,0.790,8500.000
